> **Series**: *Forecast Value*, Part 1. Part 2 trains a quantile head and a Laplace head on a simulated transformer and scores them on the definitions below. Part 3 treats overloads that last for hours.

## Value of a Forecast

An accuracy score says how far a forecast lies from what happened. It does not say what the forecast is worth to the person who acts on it. A distribution operator who must decide in the evening whether to warn a crew about tomorrow's loading of a transformer gains from a forecast only through the decision, and the decision depends on what a warning costs and on what an unwarned overload costs.[^murphy1993] Two forecasts with the same error can therefore differ in value, and a forecast with a larger error can be worth more to one user than a more accurate one.

This series measures the value of two kinds of probabilistic forecast on one decision: a quantile head, which returns a set of quantiles, and a Laplace head, which returns a location and a scale. The first part gives the definitions. It states the decision, defines the event that the decision is about, shows how each head gives the probability of the event, and defines the two scores that follow, the Brier score for the quality of the probability and the relative economic value for its worth. The illustrations use simulated numbers and are labelled as such. The data and the trained heads come in Part 2.

[^murphy1993]: A. H. Murphy, "What is a good forecast? An essay on the nature of goodness in weather forecasting," *Weather Forecast.*, vol. 8, no. 2, pp. 281-293, 1993.

In [1]:
#| output: false
import warnings

warnings.filterwarnings("ignore")

from importlib.metadata import version
import re

from IPython.display import HTML, display
from lets_plot import (
    aes, element_blank, element_line, element_rect, element_text, facet_wrap, geom_hline, geom_line, geom_point,
    geom_segment, geom_text, geom_tile, geom_vline, ggplot, ggsize, labs, scale_color_manual, scale_fill_manual, scale_linetype_manual, scale_y_discrete,
    scale_x_continuous, scale_y_continuous, theme,
)
import numpy as np
import pandas as pd
from scipy import stats

from twiga.core.metrics.decision import probability_scores, relative_economic_value_curve
from twiga.core.metrics.events import event_probability, run_event
from twiga.distributions.scenarios import quantile_cdf

# Colours by role: subject blue, highlight orange, reference slate, context grey.
BLUE, ORANGE, SLATE, INK = "#0369A1", "#E26029", "#64748B", "#1E293B"
GREY, FONT = "#CED4DA", "'Libre Franklin', 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"


def tint(colour: str, amount: float = 0.85) -> str:
    """A colour mixed with white, for bands and fills."""
    rgb = np.array([int(colour[i : i + 2], 16) for i in (1, 3, 5)])
    return "#" + "".join(f"{int(v):02X}" for v in rgb + (255 - rgb) * amount)


def house() -> theme:
    """Thin dark axes on the left and bottom, no grid, no legend, Libre Franklin type."""
    return theme(
        legend_position="none", plot_background=element_rect(fill="white", color="white"), panel_background=element_blank(),
        panel_border=element_blank(), panel_grid=element_blank(), strip_background=element_blank(),
        strip_text=element_text(face="bold", size=14, hjust=0), axis_line=element_line(size=0.5, color=INK),
        axis_ticks=element_line(size=0.5, color=INK), axis_ticks_length=4,
        text=element_text(family=FONT, size=13, color=INK), plot_margin=[8, 18, 4, 4],
    )


PHONE_WIDTH = 420
LAYOUT = {"phone": False}


def columns(desktop: int, phone: int = 1) -> int:
    """Panels in a row: ``desktop`` on a wide screen, ``phone`` in the narrow variant of a figure."""
    return phone if LAYOUT["phone"] else desktop


def size(width: int, height: int, panels: int = 1, ncol: int = 1, phone_ncol: int = 1) -> ggsize:
    """Canvas of a figure: as given on a wide screen, narrow with the panels re-stacked in the phone variant."""
    if not LAYOUT["phone"]:
        return ggsize(width, height)
    per_panel = (height - 60) / -(-panels // ncol)
    ratio = (PHONE_WIDTH / phone_ncol) / (width / ncol)
    return ggsize(PHONE_WIDTH, int(60 + -(-panels // phone_ncol) * per_panel * ratio * 1.15))


def inline(plot, kind: str) -> str:
    """The SVG of a figure as a block that scales with the page; ``kind`` marks the wide or the narrow variant."""
    svg = plot.to_svg()
    width, height = (float(v) for v in re.search(r'width="([\d.]+)" height="([\d.]+)"', svg).groups())
    svg = svg.replace(f'width="{width}" height="{height}"', f'viewBox="0 0 {width:g} {height:g}" width="100%"', 1)
    return f'<div class="letsplot letsplot-{kind}">{svg}</div>'


def show(build) -> None:
    """Draw a figure twice, wide and narrow, so that the page shows the variant that fits the width of the screen."""
    LAYOUT["phone"] = False
    wide = inline(build(), "desktop")
    LAYOUT["phone"] = True
    try:
        narrow = inline(build(), "phone")
    finally:
        LAYOUT["phone"] = False
    display(HTML(wide + narrow))


def note(frame: pd.DataFrame, colour: str = INK, hjust: float = 0.0, size: float = 5.6):
    """Direct labels from a frame with columns x, y, label (and the facet column), with a white halo over the marks."""
    return geom_text(
        aes("x", "y", label="label"), data=frame, color=colour, hjust=hjust, size=size, family=FONT,
        halo_width=2, halo_color="white", inherit_aes=False,
    )


generator = np.random.default_rng(2026)
print("twiga", version("twiga"), "| lets-plot", version("lets-plot"))

twiga 1.1.8 | lets-plot 4.11.0


## Cost-Loss Decision

The simplest account of a warning is the cost-loss model.[^murphy1977] The user can protect against an event at a cost $C$, whether or not the event happens. If the user does not protect and the event happens, the user suffers a loss $L$ with $C < L$, and if the event does not happen there is no cost. @fig-matrix shows the four outcomes. The ratio of the two is the cost-loss ratio,

$$
\alpha = \frac{C}{L}.
$$

A user who believes that the event has probability $p$ expects to lose $pL$ by not protecting and $C$ by protecting. Protecting is cheaper when $pL > C$, which is the rule

$$
p > \alpha.
$$

The rule is the best one for a user whose probabilities are reliable, and it shows why the ratio belongs to the user and not to the forecast. A crew that can be redirected at little cost has a small ratio and warns on a weak signal, while an action that is expensive and disruptive has a large ratio and waits for a strong one.

[^murphy1977]: A. H. Murphy, "The value of climatological, categorical and probabilistic forecasts in the cost-loss ratio situation," *Mon. Weather Rev.*, vol. 105, no. 7, pp. 803-816, 1977.

In [2]:
#| label: fig-matrix
#| fig-cap: "Cost of each combination of warning and event in the cost-loss model. A warning costs C whether or not the event happens, and a missed event costs L."
ALPHA_EXAMPLE = 1 / 6  # a loss six times the cost of acting
cells = pd.DataFrame(
    {
        "column": ["Event", "No event", "Event", "No event"],
        "row": ["Warning", "Warning", "No warning", "No warning"],
        "label": ["Cost C", "Cost C", "Loss L", "Zero"],
        "kind": ["act", "act", "loss", "none"],
        "meaning": ["hit", "false alarm", "miss", "correct rejection"],
    }
)
fills = {"act": tint(BLUE, 0.78), "loss": tint(ORANGE, 0.72), "none": "#F1F3F5"}
show(
    lambda: ggplot(cells, aes("column", "row"))
    + geom_tile(aes(fill="kind"), color="white", size=2.0)
    + scale_fill_manual(values=fills)
    + geom_text(aes(label="label"), data=cells.assign(row=cells["row"]), size=8.5, family=FONT, color=INK, fontface="bold", nudge_y=0.12)
    + geom_text(aes(label="meaning"), data=cells, size=6.2, family=FONT, color=SLATE, nudge_y=-0.2)
    + scale_y_discrete(limits=["No warning", "Warning"])
    + labs(x="", y="")
    + house()
    + theme(axis_line=element_blank(), axis_ticks=element_blank(), axis_text=element_text(size=13, face="bold"))
    + size(700, 260)
)

## Events as Runs

The decision turns on an event, and the event has to be defined before it can be forecast or scored. For a transformer, the quantity is the apparent power $S$ in kVA at each half-hour. A limit is set relative to the network before electric vehicles and photovoltaic generation were added, as a multiple $c$ of the 98th percentile of the apparent power $S_0$ of that network,

$$
S^{\mathrm{lim}} = c \cdot q_{0.98}(S_0).
$$

The limit measures the headroom that the network carried before distributed resources, and it is not the nameplate rating. It is chosen because it is exceeded more often as penetration grows, which keeps the event informative; an event defined by the rating can be too rare to score. The decision considered here concerns loading that persists, so the event requires that the apparent power stays above the limit for $k$ consecutive half-hours, with $k = 4$ for two hours. For a day of $H$ half-hours the outcome is

$$
o = \max_{t} \, \min_{0 \le j < k} \, \mathbb{1}\left[S_{t+j} > S^{\mathrm{lim}}\right].
$$

The outcome is one when some run of $k$ half-hours lies above the limit and zero otherwise. A value exactly at the limit is not above it, and a missing value ends a run.

## Probability from Heads

A forecast gives the probability of the event when it gives a distribution for each half-hour. The two heads give it in different ways. A quantile head returns the values $q_1 < \dots < q_K$ at levels $\tau_1 < \dots < \tau_K$ and no distribution between or beyond them.[^koenker1978] Its distribution function is completed by linear interpolation between neighbouring levels,

$$
F(y) = \tau_k + \frac{y - q_k}{q_{k+1} - q_k} \left(\tau_{k+1} - \tau_k\right),
$$

and, for $y > q_K$, by an exponential tail beyond the outermost level, whose scale $b_K$ is fixed by the two outermost quantiles on that side,

$$
F(y) = 1 - \left(1 - \tau_K\right) e^{-(y - q_K)/b_K}.
$$

The tail matters because a limit usually lies beyond the outermost level, where the quantiles are silent and a linear extension would fix the probability arbitrarily. A Laplace head returns a location $\mu$ and a scale $b$, and its distribution function is known in closed form. With the standardised distance $z = (y - \mu)/b$ it is

$$
F(y) = \frac{1}{2} + \frac{1}{2} \, \mathrm{sgn}(z) \left(1 - e^{-\lvert z \rvert}\right).
$$

For one half-hour, the probability that the apparent power exceeds the limit is one minus the distribution function at the limit,

$$
p = 1 - F\left(S^{\mathrm{lim}}\right).
$$

An event that lasts $k$ half-hours cannot be read from these marginal distributions, because it depends on how the half-hours move together. It is estimated instead from $M$ scenarios of the day, as the share of scenarios that contain the event,

$$
p = \frac{1}{M} \sum_{m=1}^{M} o^{(m)}.
$$

Part 3 constructs the scenarios. The rest of this part uses events whose probability is given directly.

In [3]:
#| label: fig-cdf
#| fig-cap: "Distribution function of one simulated half-hour from a quantile head and from a Laplace head, with the probability of exceeding the limit of 62 kVA in orange."
LEVELS = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
LIMIT = 62.0  # kVA, an illustrative headroom limit
truth = stats.gamma(a=9.0, scale=4.5)  # an apparent power at one half-hour with a right tail, mean 40 kVA
quantiles = truth.ppf(LEVELS)
scale = (quantiles[-1] - quantiles[0]) / (2 * np.log(5))  # Laplace scale that matches the 10 to 90% range
location = quantiles[4]
grid = np.linspace(8, 95, 400)
laplace_cdf = lambda y: 0.5 + 0.5 * np.sign(y - location) * (1 - np.exp(-np.abs(y - location) / scale))  # noqa: E731
curves = pd.concat(
    [
        pd.DataFrame({"y": grid, "F": quantile_cdf(grid, quantiles, LEVELS), "head": "(a) Quantile head"}),
        pd.DataFrame({"y": grid, "F": laplace_cdf(grid), "head": "(b) Laplace head"}),
    ]
)
p_quantile = float(1 - quantile_cdf(LIMIT, quantiles, LEVELS))
p_laplace = float(1 - laplace_cdf(LIMIT))
knots = pd.DataFrame({"y": quantiles, "F": LEVELS, "head": "(a) Quantile head"})
marks = pd.DataFrame(
    {
        "head": ["(a) Quantile head", "(b) Laplace head"],
        "y": [LIMIT, LIMIT],
        "F": [float(quantile_cdf(LIMIT, quantiles, LEVELS)), float(laplace_cdf(LIMIT))],
        "label": [f"p = {p_quantile:.3f}", f"p = {p_laplace:.3f}"],
    }
)
show(
    lambda: ggplot(curves, aes("y", "F"))
    + geom_line(color=BLUE, size=1.5)
    + geom_point(data=knots, shape=21, fill="white", color=BLUE, size=3.4, stroke=1.4)
    + geom_vline(xintercept=LIMIT, color=SLATE, linetype="dotted", size=0.6)
    + geom_segment(aes(x="y", xend="y", y="F", yend="yend"), data=marks.assign(yend=1.0), color=ORANGE, size=2.4)
    + note(pd.DataFrame({"head": marks["head"], "x": LIMIT + 2.5, "y": 0.62, "label": "Exceedance " + marks["label"]}), INK, 0.0, 5.8)
    + note(pd.DataFrame({"head": marks["head"], "x": LIMIT - 1.5, "y": 0.06, "label": "Limit"}), SLATE, 1.0, 5.4)
    + facet_wrap("head", ncol=columns(2))
    + scale_x_continuous(name="Apparent power (kVA)")
    + scale_y_continuous(name="Probability that power is at most y", limits=[0, 1.02], breaks=[0, 0.25, 0.5, 0.75, 1.0])
    + house()
    + size(900, 360, 2, 2)
)

@fig-cdf applies both heads to one half-hour of a simulated transformer whose apparent power has a right tail. The true probability of exceeding the limit of 62 kVA is 0.069. The quantile head gives 0.071, with its nine quantiles drawn as hollow points and the exponential tail carrying the curve beyond the last one. The Laplace head gives 0.057, which is 18% below the true value. The Laplace distribution is symmetric about its median, so a right tail that is heavier than a Laplace tail is understated, and an operator who warned at a ratio of 0.06 would warn under the quantile head and not under the Laplace head. The two heads can therefore disagree about the same day, and the scores below decide which disagreement costs more.

## Scoring a Probability

The quality of a probability of an event is measured by the Brier score, the mean squared difference between the forecast probability $p_n$ and the outcome $o_n \in \{0, 1\}$ over $N$ days,[^brier1950]

$$
\mathrm{BS} = \frac{1}{N} \sum_{n=1}^{N} (p_n - o_n)^2.
$$

The score is strictly proper, so the best expected score is reached only by stating the true probability.[^gneiting2007] It separates into three parts,[^murphy1973]

$$
\mathrm{BS} = \mathrm{REL} - \mathrm{RES} + \mathrm{UNC}.
$$

The reliability REL measures the gap between the stated probabilities and the frequencies with which the event followed them, and a reliable forecast has a small REL. The resolution RES measures how far the frequencies of the forecast classes lie from the overall rate, and a forecast that separates the days with the event from the days without has a large RES. The uncertainty UNC depends only on the event rate, so it is the same for every forecast. The skill compares the score with that of a reference that states a constant rate $\bar{s}$ taken from the training window,

$$
\mathrm{BSS} = 1 - \frac{\mathrm{BS}}{\mathrm{BS}_{\mathrm{ref}}}.
$$

A skill of one is a perfect forecast, zero is no better than the reference, and a negative value is worse. A constant rate is a weak reference where the risk depends on the hour of the day, and the later parts also use the rate for each hour.

[^brier1950]: G. W. Brier, "Verification of forecasts expressed in terms of probability," *Mon. Weather Rev.*, vol. 78, no. 1, pp. 1-3, 1950.

[^gneiting2007]: T. Gneiting and A. E. Raftery, "Strictly proper scoring rules, prediction, and estimation," *J. Amer. Statist. Assoc.*, vol. 102, no. 477, pp. 359-378, 2007.

[^murphy1973]: A. H. Murphy, "A new vector partition of the probability score," *J. Appl. Meteor.*, vol. 12, no. 4, pp. 595-600, 1973.

In [4]:
#| output: false
DAYS, TRAIN = 6000, 3000
risk = generator.beta(2.0, 6.0, DAYS)  # the true event probability of each day, mean 0.25
event = (generator.uniform(size=DAYS) < risk).astype(int)
logit = np.log(risk / (1 - risk))
forecasts = {
    "Calibrated": risk,
    "Over-confident": 1 / (1 + np.exp(-2.2 * logit)),  # the same ranking of days, probabilities pushed to the extremes
}
reference_rate = float(event[:TRAIN].mean())  # the rate a forecaster could have seen before the test days
test = slice(TRAIN, DAYS)
rows, curves_value = [], []
for name, probability in forecasts.items():
    scores = probability_scores(probability[test], event[test], reference_rate=reference_rate)
    ratios, value = relative_economic_value_curve(probability[test], event[test], reference_rate=reference_rate)
    rows.append({"forecast": name, **{k: round(v, 4) for k, v in scores.items() if k in ("brier_score", "brier_skill_score", "reliability", "resolution", "uncertainty")}, "best_value": round(float(np.nanmax(value)), 3), "at_ratio": round(float(ratios[np.nanargmax(value)]), 2)})
    curves_value.append(pd.DataFrame({"ratio": ratios, "value": value, "forecast": name}))
summary = pd.DataFrame(rows)
curves_value = pd.concat(curves_value)
print(f"reference rate {reference_rate:.3f}; test rate {event[test].mean():.3f}")
print(summary.to_string(index=False))

reference rate 0.258; test rate 0.247
      forecast  brier_score  brier_skill_score  reliability  resolution  uncertainty  best_value  at_ratio
    Calibrated       0.1678             0.0978       0.0002      0.0173       0.1858       0.288      0.25
Over-confident       0.1840             0.0106       0.0154      0.0154       0.1858       0.224      0.25


A simulated example shows what the three parts measure. Over 6,000 days the true probability of the event follows a Beta distribution with mean 0.25, and the outcome of each day is drawn from it. Two forecasts are scored on the last 3,000 days, with the rate of the first 3,000 as the reference. The calibrated forecast states the true probability. The over-confident forecast ranks the days in the same order but pushes its probabilities towards zero and one. The event rate on the test days is 0.247.

The calibrated forecast has a Brier score of 0.1678 and a skill of 0.098. Its reliability is 0.0002, so its probabilities match the frequencies, and its resolution of 0.0173 is the information that it adds to the event rate. The over-confident forecast has a Brier score of 0.1840 and a skill of 0.011. Its resolution of 0.0154 is similar, so it separates the days almost as well, but its reliability of 0.0154 is far larger than the 0.0002 of the calibrated forecast. The whole of its loss lies in the stated probabilities and none in the ranking of the days. A user who acts on these probabilities, as the next section shows, pays for that loss. The three parts are computed in ten equal probability bins, so they add to the score only approximately.

## Relative Economic Value

The value of a probability forecast to a user is the expense saved by acting on it, set against what the user would spend without it. Let a warning be issued when the probability exceeds $\alpha$, and let $a$ be the days with a warning and an event, $b$ the days with a warning and no event, and $c$ the days with an event and no warning, among $N$ days. In units of $L$ the user who follows the forecast pays

$$
E_{\mathrm{fc}} = \frac{\alpha (a + b) + c}{N}.
$$

Without a forecast the user either always protects or never protects. With the event frequency $s$, the better of the two pays

$$
E_{\mathrm{clim}} = \min(\alpha, s),
$$

and a user with perfect knowledge protects only on the days with an event and pays

$$
E_{\mathrm{perf}} = \alpha s.
$$

The relative economic value is the share of the possible saving that the forecast achieves,[^richardson2000]

$$
V(\alpha) = \frac{E_{\mathrm{clim}} - E_{\mathrm{fc}}}{E_{\mathrm{clim}} - E_{\mathrm{perf}}}.
$$

It is one for a perfect forecast and zero for the better of always and never protecting, and it is negative where the forecast does worse than that. Consider 100 days with 20 events and a ratio of $\alpha = 1/6$, so that a loss is six times the cost of acting. A forecast warns on 30 days, with 16 hits, 14 false alarms and 4 misses. Then $E_{\mathrm{fc}} = (30/6 + 4)/100 = 0.090$, $E_{\mathrm{clim}} = 1/6 = 0.167$ and $E_{\mathrm{perf}} = 0.033$, so that $V = 0.077 / 0.133 = 0.575$. The forecast achieves 58% of the saving that perfect knowledge would give this user.

[^richardson2000]: D. S. Richardson, "Skill and relative economic value of the ECMWF ensemble prediction system," *Q. J. R. Meteorol. Soc.*, vol. 126, no. 563, pp. 649-667, 2000.

In [5]:
#| output: false
# The worked example of the text: 100 days, 20 events, 30 warnings of which 16 are hits, 14 false alarms, 4 misses.
N_DAYS, hits, false_alarms, misses = 100, 16, 14, 4
events = hits + misses
s = events / N_DAYS
expense_forecast = (ALPHA_EXAMPLE * (hits + false_alarms) + misses) / N_DAYS
expense_climatology = min(ALPHA_EXAMPLE, s)
expense_perfect = ALPHA_EXAMPLE * s
value = (expense_climatology - expense_forecast) / (expense_climatology - expense_perfect)
hit_rate, false_alarm_rate = hits / events, false_alarms / (N_DAYS - events)
form_book = (expense_climatology - false_alarm_rate * ALPHA_EXAMPLE * (1 - s) - hit_rate * ALPHA_EXAMPLE * s - (1 - hit_rate) * s) / (expense_climatology - ALPHA_EXAMPLE * s)
print(f"E_fc {expense_forecast:.4f}, E_clim {expense_climatology:.4f}, E_perf {expense_perfect:.4f}, V {value:.4f}; hit-rate form {form_book:.4f}")
exact = float(truth.sf(LIMIT))
over = curves_value[curves_value["forecast"] == "Over-confident"].dropna()
calibrated = curves_value[curves_value["forecast"] == "Calibrated"].dropna()
print(f"exact exceedance {exact:.4f}, quantile {p_quantile:.4f}, laplace {p_laplace:.4f}")
print("over-confident value first non-negative at ratio", float(over.loc[over["value"] >= 0, "ratio"].iloc[0]))
print("calibrated value above 0.05 up to ratio", float(calibrated.loc[calibrated["value"] > 0.05, "ratio"].max()))
print("over-confident value at ratios 0.1 and 0.15:", [round(float(over.loc[(over["ratio"] - r).abs().idxmin(), "value"]), 3) for r in (0.1, 0.15)])

E_fc 0.0900, E_clim 0.1667, E_perf 0.0333, V 0.5750; hit-rate form 0.5750
exact exceedance 0.0691, quantile 0.0715, laplace 0.0568
over-confident value first non-negative at ratio 0.19
calibrated value above 0.05 up to ratio 0.47000000000000003
over-confident value at ratios 0.1 and 0.15: [-0.485, -0.144]


In [6]:
#| label: fig-value
#| fig-cap: "Relative economic value of two simulated forecasts against the cost-loss ratio. The over-confident forecast has a negative value for every ratio below 0.19."
best = curves_value.loc[curves_value.groupby("forecast")["value"].idxmax()]
plotted = curves_value[curves_value["value"] >= -0.6]  # clipped to the axis, so that no row is dropped by the scale
show(
    lambda: ggplot(plotted, aes("ratio", "value", color="forecast"))
    + geom_hline(yintercept=0.0, color=SLATE, linetype="dashed", size=0.6)
    + geom_line(size=1.5)
    + scale_color_manual(values={"Calibrated": BLUE, "Over-confident": ORANGE})
    + geom_point(data=best, shape=21, fill="white", size=3.6, stroke=1.5)
    + note(pd.DataFrame({"x": [0.29], "y": [0.35], "label": ["Calibrated"]}), BLUE, 0.0, 6.0)
    + note(pd.DataFrame({"x": [0.12], "y": [-0.42], "label": ["Over-confident"]}), ORANGE, 0.0, 6.0)
    + note(pd.DataFrame({"x": [0.99], "y": [-0.1], "label": ["Climatology: no value"]}), SLATE, 1.0, 5.4)
    + scale_x_continuous(name="Cost-loss ratio", limits=[0, 1], breaks=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
    + scale_y_continuous(name="Relative economic value", limits=[-0.6, 0.6], breaks=[-0.5, 0, 0.5])
    + house()
    + size(900, 400)
)

@fig-value gives the value of the two simulated forecasts for every ratio. The calibrated forecast has a value of at least 0.05 for ratios up to 0.47, with a maximum of 0.288 at a ratio of 0.25. The maximum lies near the event rate of 0.247. For ratios above 0.5 the value is close to zero. The over-confident forecast has a maximum of 0.224 at the same ratio, which is 22% lower, and it has a negative value for every ratio below 0.19, falling to $-0.49$ at a ratio of 0.10. A user with a cheap action would have done better by ignoring the forecast and following the better of always or never protecting.

The Brier score and the value therefore agree on the ranking here and disagree on how much it matters. The skill of the over-confident forecast, 0.011, says that it is about as good as the constant rate, and the value curve says for whom it is worse. The users who lose most are those with the smallest cost-loss ratios. A forecast is judged by the users that it is meant to serve, and the two heads of Part 2 are compared on that curve.

## Limits of the Illustration

- **The numbers are simulated.** The probabilities, the outcomes and the apparent power of @fig-cdf are drawn from stated distributions to show what the scores measure. They are not results for a transformer.
- **The ratio belongs to the user.** The value curve is a family of users, and a single number for the value of a forecast does not exist without a ratio. Part 2 reports the curve and the ratios at which a head is best.
- **The reference is a constant rate.** It is a weak reference for risk that depends on the hour of the day, and a skill against it can look larger than a skill against the rate for each hour.
- **Days are treated as independent.** The probabilities of consecutive days are correlated, which widens the uncertainty of every score in this series beyond what the number of days suggests.

## Next Steps

Part 2 takes a simulated transformer with a high penetration of electric vehicles and photovoltaic generation, defines the headroom limit from its pre-DER record, trains a quantile head and a Laplace head, and scores the probability of the single-step event with the Brier score and the value curve defined here. Part 3 constructs scenarios for the event that lasts two hours, shows how much a forecast that treats the half-hours as independent underestimates it, and measures what the dependence adds to the value.

## Reproducible Results

The notebook computes every figure of this part from a seeded generator and the functions of twiga that score events and their value, `twiga.core.metrics.events` and `twiga.core.metrics.decision`. The event functions are in the main branch of twiga and are not yet in a release, so a release of twiga that contains them is needed to reproduce the notebook from the package index.